# PyBullet Squishy Object Drop: Multi-View Video Capture

This notebook uses `phys_sim`, PyBullet's deformable-world solver, and TinyRenderer to simulate a squishy soft object dropping onto a tabletop and deforming under impact. It records front, side, back, and top videos plus synchronized deformation metrics from the simulated soft mesh.

In [ ]:
import sys
from pathlib import Path

assert "phys_sim" in sys.executable, (
    f"Expected the phys_sim virtual environment, but got: {sys.executable}\n"
    "In Jupyter, select the kernel named 'phys_sim'."
)

import numpy as np
import pybullet as p
import pybullet_data
import imageio.v2 as imageio
from IPython.display import Video, display

print(f"Python executable: {sys.executable}")
print(f"PyBullet data path: {pybullet_data.getDataPath()}")


## Scenario Parameters

The deformable object is a soft torus/ring mesh from local `pybullet_data`. It is dropped under gravity onto the tabletop. Soft-body mass, elastic stiffness, damping, friction, and collision margin are explicit so the scenario can be tuned. This is a surface soft-body approximation, but the impact and flattening are observable and physically driven by PyBullet contact forces.

In [ ]:
OUTPUT_DIR = Path("pybullet_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

GRAVITY = -9.80665
SIM_HZ = 240
TIME_STEP = 1.0 / SIM_HZ
VIDEO_FPS = 60
STEPS_PER_FRAME = SIM_HZ // VIDEO_FPS
DURATION_SEC = 2.6
N_STEPS = int(DURATION_SEC * SIM_HZ)

TABLE_LENGTH = 1.20
TABLE_WIDTH = 0.80
TABLE_THICKNESS = 0.06
TABLE_TOP_Z = 0.75
TABLE_CENTER_Z = TABLE_TOP_Z - TABLE_THICKNESS / 2.0
TABLE_RESTITUTION = 0.05
TABLE_FRICTION = 0.80

SOFT_MASS = 0.18
SOFT_SCALE = 0.22
SOFT_BASE_POSITION = [0.0, 0.0, TABLE_TOP_Z + 0.50]
SOFT_ELASTIC_STIFFNESS = 10
SOFT_DAMPING = 0.25
SOFT_FRICTION = 0.90
SOFT_COLLISION_MARGIN = 0.006

IMG_WIDTH = 640
IMG_HEIGHT = 368

print(f"Writing outputs to: {OUTPUT_DIR.resolve()}")


In [ ]:
def connect_pybullet(deformable=False):
    if p.isConnected():
        p.disconnect()
    client = p.connect(p.DIRECT)
    if deformable:
        p.resetSimulation(p.RESET_USE_DEFORMABLE_WORLD, physicsClientId=client)
    else:
        p.resetSimulation(physicsClientId=client)
    p.setAdditionalSearchPath(pybullet_data.getDataPath(), physicsClientId=client)
    p.setGravity(0, 0, GRAVITY, physicsClientId=client)
    p.setTimeStep(TIME_STEP, physicsClientId=client)
    p.setPhysicsEngineParameter(
        fixedTimeStep=TIME_STEP,
        numSolverIterations=120,
        numSubSteps=1,
        deterministicOverlappingPairs=1,
        physicsClientId=client,
    )
    return client


def create_table(client):
    p.loadURDF("plane.urdf", physicsClientId=client)
    collision = p.createCollisionShape(
        p.GEOM_BOX,
        halfExtents=[TABLE_LENGTH / 2, TABLE_WIDTH / 2, TABLE_THICKNESS / 2],
        physicsClientId=client,
    )
    visual = p.createVisualShape(
        p.GEOM_BOX,
        halfExtents=[TABLE_LENGTH / 2, TABLE_WIDTH / 2, TABLE_THICKNESS / 2],
        rgbaColor=[0.58, 0.40, 0.24, 1.0],
        physicsClientId=client,
    )
    table_id = p.createMultiBody(
        baseMass=0,
        baseCollisionShapeIndex=collision,
        baseVisualShapeIndex=visual,
        basePosition=[0, 0, TABLE_CENTER_Z],
        physicsClientId=client,
    )
    p.changeDynamics(
        table_id,
        -1,
        restitution=TABLE_RESTITUTION,
        lateralFriction=TABLE_FRICTION,
        spinningFriction=0.01,
        rollingFriction=0.01,
        physicsClientId=client,
    )
    return table_id

def camera_matrices(view_name, target):
    cameras = {
        "front": {"eye": [0.0, -1.65, 1.08], "up": [0, 0, 1], "fov": 50},
        "side": {"eye": [1.65, 0.0, 1.08], "up": [0, 0, 1], "fov": 50},
        "back": {"eye": [0.0, 1.65, 1.08], "up": [0, 0, 1], "fov": 50},
        "top": {"eye": [0.0, 0.0, 2.35], "up": [0, 1, 0], "fov": 44},
    }
    spec = cameras[view_name]
    view = p.computeViewMatrix(spec["eye"], target, spec["up"])
    proj = p.computeProjectionMatrixFOV(
        fov=spec["fov"],
        aspect=IMG_WIDTH / IMG_HEIGHT,
        nearVal=0.02,
        farVal=5.0,
    )
    return view, proj


CAMERA_NAMES = ["front", "side", "back", "top"]


def render_rgb(client, view_name, target):
    view, proj = camera_matrices(view_name, target)
    _, _, rgba, _, _ = p.getCameraImage(
        width=IMG_WIDTH,
        height=IMG_HEIGHT,
        viewMatrix=view,
        projectionMatrix=proj,
        renderer=p.ER_TINY_RENDERER,
        lightDirection=[-0.5, -0.4, -1.0],
        physicsClientId=client,
    )
    rgba = np.asarray(rgba, dtype=np.uint8).reshape(IMG_HEIGHT, IMG_WIDTH, 4)
    return rgba[:, :, :3]


def make_writers(prefix):
    paths = {name: OUTPUT_DIR / f"{prefix}_{name}.mp4" for name in CAMERA_NAMES}
    writers = {
        name: imageio.get_writer(path, fps=VIDEO_FPS, codec="libx264", quality=8, macro_block_size=16)
        for name, path in paths.items()
    }
    return paths, writers

def create_deformable_scene(client):
    p.setPhysicsEngineParameter(sparseSdfVoxelSize=0.03, physicsClientId=client)
    table_id = create_table(client)
    soft_id = p.loadSoftBody(
        "torus/torus_textured.obj",
        basePosition=SOFT_BASE_POSITION,
        scale=SOFT_SCALE,
        mass=SOFT_MASS,
        useNeoHookean=0,
        useBendingSprings=1,
        useMassSpring=1,
        springElasticStiffness=SOFT_ELASTIC_STIFFNESS,
        springDampingStiffness=SOFT_DAMPING,
        springDampingAllDirections=1,
        useSelfCollision=0,
        frictionCoeff=SOFT_FRICTION,
        collisionMargin=SOFT_COLLISION_MARGIN,
        physicsClientId=client,
    )
    p.changeVisualShape(soft_id, -1, rgbaColor=[0.10, 0.52, 0.88, 1.0], physicsClientId=client)
    return table_id, soft_id


def soft_mesh_stats(client, soft_id):
    _, vertices = p.getMeshData(soft_id, flags=p.MESH_DATA_SIMULATION_MESH, physicsClientId=client)
    vertices = np.asarray(vertices, dtype=float)
    z_min = float(vertices[:, 2].min())
    z_max = float(vertices[:, 2].max())
    center = vertices.mean(axis=0)
    return z_min, z_max, center


## Run Simulation and Record Videos

The soft mesh statistics are sampled once per rendered video frame, giving a synchronized deformation trace.

In [ ]:
client = connect_pybullet(deformable=True)
table_id, soft_id = create_deformable_scene(client)
video_paths, writers = make_writers("deformable_impact")
records = []

try:
    for step in range(N_STEPS + 1):
        t = step * TIME_STEP
        if step % STEPS_PER_FRAME == 0:
            z_min, z_max, soft_center = soft_mesh_stats(client, soft_id)
            _, vertices = p.getMeshData(soft_id, flags=p.MESH_DATA_SIMULATION_MESH, physicsClientId=client)
            vertices = np.asarray(vertices, dtype=float)
            xy_radius = np.max(np.linalg.norm(vertices[:, :2] - soft_center[:2], axis=1))
            records.append([t, z_min, z_max, z_max - z_min, xy_radius, soft_center[0], soft_center[1], soft_center[2]])
            target = [0.0, 0.0, TABLE_TOP_Z + 0.13]
            for name, writer in writers.items():
                writer.append_data(render_rgb(client, name, target))

        p.stepSimulation(physicsClientId=client)
finally:
    for writer in writers.values():
        writer.close()
    p.disconnect(client)

records = np.asarray(records)
trajectory_path = OUTPUT_DIR / "deformable_impact_metrics.csv"
np.savetxt(
    trajectory_path,
    records,
    delimiter=",",
    header="time,soft_z_min,soft_z_max,soft_z_span,soft_xy_radius,soft_center_x,soft_center_y,soft_center_z",
    comments="",
)

print("Saved videos:")
for name, path in video_paths.items():
    print(f"  {name:>5}: {path}")
print(f"Saved deformation metrics: {trajectory_path}")


## Quick Deformation Checks

In [ ]:
initial_z_span = records[0, 3]
min_z_span = records[:, 3].min()
initial_xy_radius = records[0, 4]
max_xy_radius = records[:, 4].max()
min_soft_z = records[:, 1].min()
impact_frame = records[np.argmin(records[:, 1])]
height_compression = initial_z_span - min_z_span
radial_spread = max_xy_radius - initial_xy_radius

print(f"Initial object height span: {initial_z_span:.4f} m")
print(f"Minimum height span:        {min_z_span:.4f} m")
print(f"Height compression:         {height_compression:.4f} m")
print(f"Initial xy radius:          {initial_xy_radius:.4f} m")
print(f"Maximum xy radius:          {max_xy_radius:.4f} m")
print(f"Radial spread:              {radial_spread:.4f} m")
print(f"Minimum soft vertex z:      {min_soft_z:.4f} m")
print(f"Lowest-frame time:          {impact_frame[0]:.3f} s")
print("Compression plus radial spread indicates visible squishing under impact.")


## Preview Videos

In [ ]:
for name in CAMERA_NAMES:
    print(name)
    display(Video(str(video_paths[name]), embed=True, html_attributes="controls loop"))
